In [2]:
import numpy as np

X = np.load("../src/X_data.npy")
y = np.load("../src/y_user.npy")

print(X.shape)
print(y.shape)

(1724, 100, 64)
(1724,)


In [3]:
X_normalized = []

for sample in X:

    sample_mean = np.mean(sample)
    sample_std = np.std(sample)

    sample = (
        sample - sample_mean
    ) / (
        sample_std + 1e-8
    )

    X_normalized.append(sample)

X_normalized = np.array(
    X_normalized,
    dtype=np.float32
)

print(X_normalized.shape)

print(np.mean(X_normalized))
print(np.std(X_normalized))

(1724, 100, 64)
-6.084929e-11
1.0


In [4]:
selected_classes = [0,4]

mask = np.isin(
    y,
    selected_classes
)

X_binary = X_normalized[mask]
y_binary = y[mask]

print(X_binary.shape)

(737, 100, 64)


In [5]:
y_binary = np.where(
    y_binary == 0,
    0,
    1
)

print(np.unique(y_binary))

[0 1]


In [6]:
aleyna_idx = np.where(
    y_binary == 0
)[0]

empty_idx = np.where(
    y_binary == 1
)[0]

np.random.seed(42)

selected_aleyna = np.random.choice(
    aleyna_idx,
    size=160,
    replace=False
)

selected_idx = np.concatenate([
    selected_aleyna,
    empty_idx
])

X_balanced = X_binary[
    selected_idx
]

y_balanced = y_binary[
    selected_idx
]

print(
    np.bincount(y_balanced)
)

[160 160]


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_balanced,
    y_balanced,
    test_size=0.20,
    random_state=42,
    stratify=y_balanced
)

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dense,
    Dropout
)

model = Sequential([

    Conv1D(
        32,
        3,
        activation="relu",
        input_shape=(100,64)
    ),

    MaxPooling1D(2),

    Conv1D(
        64,
        3,
        activation="relu"
    ),

    MaxPooling1D(2),

    LSTM(
        64
    ),

    Dropout(0.5),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        2,
        activation="softmax"
    )
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

I0000 00:00:1780748277.172488   45471 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780748277.279479   45471 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780748280.218633   45471 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/aleynagul/csi-gesture-identity-recognition/csi_env/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1780748281.476859   45471 cuda_platform.cc:52] failed 

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 98, 32)         │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 49, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 47, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 23, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,554 (185.76 KB)

 Trainable params: 47,554 (185.76 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32
)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 125ms/step - accuracy: 0.5245 - loss: 0.7197 - val_accuracy: 0.4615 - val_loss: 0.6952
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5686 - loss: 0.6871 - val_accuracy: 0.4423 - val_loss: 0.6872
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6078 - loss: 0.6633 - val_accuracy: 0.5769 - val_loss: 0.6636
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.6961 - loss: 0.6188 - val_accuracy: 0.5769 - val_loss: 0.6304
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.7402 - loss: 0.5204 - val_accuracy: 0.7500 - val_loss: 0.5888
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.7843 - loss: 0.4617 - val_accuracy: 0.7308 - val_loss: 0.4999
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.7598 - loss: 0.4795 - val_accuracy: 0.7885 - val_loss: 0.4983
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.7990 - loss: 0.4330 - val_accuracy: 0.7692 - val_loss: 0.5107

In [10]:
model.evaluate(X_test, y_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.7344 - loss: 0.6554


[0.6554245948791504, 0.734375]

In [11]:
from sklearn.metrics import classification_report
import numpy as np

pred = model.predict(X_test)

pred_class = np.argmax(pred, axis=1)

print(
    classification_report(
        y_test,
        pred_class,
        target_names=[
            "aleyna",
            "empty"
        ]
    )
)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step 
              precision    recall  f1-score   support

      aleyna       0.76      0.69      0.72        32
       empty       0.71      0.78      0.75        32

    accuracy                           0.73        64
   macro avg       0.74      0.73      0.73        64
weighted avg       0.74      0.73      0.73        64

